# 2 · An address book: relations instead of lists

**The problem.** A contact has several addresses, each with a label ("home", "work"). The obvious model is a list
of address records nested inside the contact. Then real data shows up:

- Alice and her husband share a home address. With nesting it's stored twice, and fixing a typo in one copy leaves
  the other wrong.
- "Who lives at 10 Downing Street?" means scanning every contact's list.
- The label belongs to neither side. It isn't a property of the address (the same flat is "home" for Alice and
  "office" for her lodger), and it isn't a property of the contact.

The framework has **no lists and no nested records**. Every collection is a **relation**: a set of *entries*, each
linking objects together and optionally carrying its own properties (like the label). Anything a list, dictionary
or pointer does, a relation does too. Unlike them, a relation is visible from every object it links.

In [1]:
from mbse.Schemas.Framework import Proxies, Schemas
from toolkit import entries

## Step 1: the objects

First, the things that exist on their own: contacts and addresses. There's nothing new here yet.

In [2]:
Contact = (
    Schemas.OfObject.Builder().ref()
    .properties(
        lambda prop: prop.name('given_name').of(lambda t: t.as_native(str)),
        lambda prop: prop.name('family_name').of(lambda t: t.as_native(str)),
    )
    .create()
)

Address = (
    Schemas.OfObject.Builder().ref()
    .properties(
        lambda prop: prop.name('street').of(lambda t: t.as_native(str)),
        lambda prop: prop.name('locality').of(lambda t: t.as_native(str)),
        lambda prop: prop.name('postal_code').of(lambda t: t.as_native(str)),
        lambda prop: prop.name('country_code').of(lambda t: t.as_native(str)),  # ISO 3166-1 alpha-2
    )
    .create()
)

## Step 2: the relation

A relation names its **links** (one per object an entry connects) and its own **properties**. Notice that the links
have no types. `ContactAddresses` doesn't say *what* fills `contact` or `address`. It only says an entry has two
ends with those names, plus a `label`.

In [3]:
ContactAddresses = (
    Schemas.OfRelation.Builder()
    .links('contact', 'address')
    .properties(lambda prop: prop.name('label').of(lambda t: t.as_native(str)))  # 'home', 'work', ...
    .create()
)

## Step 3: each object declares its side

The objects say which relations they take part in, and through which link. That's an *adjacency*:

- `name` is the accessor you'll use on the object (`.addresses(...)`),
- `of` is the relation,
- `me` is the link this object fills.

The schemas already exist, so they are edited in place with a builder started from them, finished with `update()`.
You saw the same `update()` on instances in case study 1.

In [4]:
Contact = (
    Schemas.OfObject.Builder(Contact)
    .relations(lambda adj: adj.name('addresses').of(ContactAddresses).me('contact'))
    .update()
)
Address = (
    Schemas.OfObject.Builder(Address)
    .relations(lambda adj: adj.name('residents').of(ContactAddresses).me('address'))
    .update()
)
assert Contact.validate() == [] and Address.validate() == [] and ContactAddresses.validate() == []

### Why three steps?

The contact schema mentions the relation, and the relation's meaning depends on the objects on each side. Something
has to exist first. Keeping links untyped and letting each object declare its own adjacencies breaks the cycle:

1. objects, 2. relations, 3. adjacencies added with `Builder(existing)...update()`.

This is also why building never validates: halfway through step 3 the schemas are legitimately incomplete.

The same untyped link can be filled by more than one kind of object. A company could declare `.me('contact')` on
`ContactAddresses` too. Case study 4 comes back to what that means.

## Registering

Relations are registered too, because serialized data will name them. There is no builder for a relation, though.
You'll see why in a moment.

In [5]:
Proxies.register('Contact', Contact)
Proxies.register('Address', Address)
Proxies.register('ContactAddresses', ContactAddresses)
Builders = Proxies.Builders

try:
    Builders.ContactAddresses()
except TypeError as error:
    print('TypeError:', error)

TypeError: 'ContactAddresses' is a relation; no relation builder is exposed


## Adding entries

Entries are added *from an object*, through its adjacency accessor. The accessor takes an **entry Spec**: a function
that receives an *entry builder* `x`. The object fills its own link (`contact`) by itself. You set the other links
and the entry's properties, fluently, with the same one-setter-per-name style as instances:

In [6]:
downing = (
    Builders.Address()
    .street('10 Downing Street').locality('London').postal_code('SW1A 2AA').country_code('GB')
    .create()
)

alice = (
    Builders.Contact()
    .given_name('Alice')
    .family_name('Liddell')
    .addresses(lambda x: x.address(downing).label('home'))
    .create()
)
row, = entries(alice, 'addresses')
print(row['address'].street, '|', row['label'])

10 Downing Street | home


`entries` is a small helper, explained next. It returns one dict per entry, mapping each link to the linked object
and each property to its value.

This is why relations have no builder of their own: an entry always belongs to the objects it links, so it's
created from one of them, which fills its own end.

### An aside: how `entries` reads data

Instances expose properties as attributes, but their entries aren't attributes: an entry is shared by several
objects, so it doesn't belong to any one of them. You read entries the same way *any* schema-agnostic tool reads
data, by **visiting**. A builder started from an instance is a visitor over that instance's data. Its methods don't
return values. They pass *handles* to callbacks:

```python
Builders.Contact(alice).adjacency('addresses', lambda a:        # a: the adjacency
    a.entries(lambda e:                                        # e: one entry
        e.links(lambda k: k.target(print))))                   # k: one link; target: the object
```

Callbacks instead of return values let the same protocol drive readers, writers, serializers and validators
without building intermediate lists. The helper `entries` in `toolkit.py` wraps this into a list of dicts. It's
short enough to read in full:

In [7]:
import inspect
import toolkit

print(inspect.getsource(toolkit.entries))

def entries(obj, adjacency):
    """The entries of one of `obj`'s adjacencies, as dicts of link and property values. (Case study 2.)

    The object's own link is implied, so it is not included. Reading goes through a builder started from `obj`: the
    builder is a visitor, and visiting is how anything is read without knowing the schema in advance. The builder is
    never finalized, so nothing changes.
    """
    rows = []

    def read(entry):
        row = {}
        entry.links(lambda link: link.target(lambda target: row.__setitem__(link.name(), target)))
        entry.properties(
            lambda prop: prop.value(lambda v: v.as_native(lambda n: row.__setitem__(prop.name(), n.get())))
        )
        rows.append(row)

    getattr(Proxies.Builders, obj.schema_name())(obj).adjacency(adjacency, lambda a: a.entries(read))
    return rows



## Shared addresses: one entry, two views

Now the husband, Reginald, shares the same address object. With nested lists this would be a copy. Here it's
another entry pointing at the same address:

In [8]:
reggie = (
    Builders.Contact()
    .given_name('Reginald')
    .family_name('Hargreaves')
    .addresses(lambda x: x.address(downing).label('home'))
    .create()
)

print('residents:', [(e['contact'].given_name, e['label']) for e in entries(downing, 'residents')])

residents: [('Alice', 'home'), ('Reginald', 'home')]


"Who lives here?" needed no scan and no second index. Each entry is **one fact** seen from both of its ends: Alice's
`addresses` and the address's `residents` are two views of the same entries.

## Creating the other end inline

A link takes an existing object *or* a Spec that builds a new one on the spot:

In [9]:
Builders.Contact(alice).addresses(
    lambda x: x.address(lambda a: a.street('1 Infinite Loop').locality('Cupertino').country_code('US')).label('work')
).update()

[(e['address'].street, e['label']) for e in entries(alice, 'addresses')]

[('10 Downing Street', 'home'), ('1 Infinite Loop', 'work')]

How did the framework know the new object is an `Address`? The link is untyped, but exactly one registered schema
declares an adjacency filling `address` in `ContactAddresses`. That inference must be unambiguous. If two schemas
filled the link, inline creation would be refused and you'd pass an existing object instead (case study 4 shows
this).

## Entries form a set

Adding an entry equal to an existing one does nothing. Equal means the same linked objects (by identity) and equal
property values. An entry with a different label is a different fact, and so is one with no label at all:

In [10]:
Builders.Contact(reggie).addresses(lambda x: x.address(downing).label('home')).update()    # same fact: elided
Builders.Contact(reggie).addresses(lambda x: x.address(downing).label('office')).update()  # a different fact
Builders.Contact(reggie).addresses(lambda x: x.address(downing)).update()                  # no label: different again
sorted(e.get('label', '(no label)') for e in entries(reggie, 'addresses'))

['(no label)', 'home', 'office']

**Why a set?** A list could hold the same fact twice, and then removing "it" or counting "them" becomes ambiguous.
A set can't. If you really want two otherwise identical entries, the relation is missing a property that tells them
apart, such as a `since` date or a sequence number.

## Phones: why a one-link relation is not allowed

Phone numbers are the next thing to model, and the tempting shortcut is a relation with a single link whose
entries hold the number. The framework rejects that. An entry that carries data but links only one object is really
an object pretending to be a relation. Model the phone number as an object and link it:

In [11]:
Shortcut = (
    Schemas.OfRelation.Builder()
    .links('contact')
    .properties(lambda prop: prop.name('number').of(lambda t: t.as_native(str)))
    .create()
)
Shortcut.validate()

['a relation needs at least two links; a one-link relation merges a relation and an object']

In [12]:
PhoneNumber = Schemas.OfObject.Builder().ref().properties(lambda prop: prop.name('number').of(lambda t: t.as_native(str))).create()
ContactPhones = (
    Schemas.OfRelation.Builder()
    .links('contact', 'phone')
    .properties(lambda prop: prop.name('label').of(lambda t: t.as_native(str)))
    .create()
)
PhoneNumber = Schemas.OfObject.Builder(PhoneNumber).relations(lambda adj: adj.name('contacts').of(ContactPhones).me('phone')).update()
Contact = Schemas.OfObject.Builder(Contact).relations(lambda adj: adj.name('phones').of(ContactPhones).me('contact')).update()
Proxies.register('PhoneNumber', PhoneNumber)
Proxies.register('ContactPhones', ContactPhones)

Builders.Contact(alice).phones(lambda x: x.phone(lambda p: p.number('+447700900123')).label('mobile')).update()
[(e['phone'].number, e['label']) for e in entries(alice, 'phones')]

[('+447700900123', 'mobile')]

Two things happened there. Adding `phones` to `Contact` with `update()` changed the registered schema *in place*,
so existing contacts gained the new adjacency. And the switchboard number a whole office shares is now one object
linked from many contacts, just like the shared address.

## The rules you met

Mistakes in entries are caught while you build them:

In [13]:
for attempt, why in [
    (lambda: Builders.Contact().addresses(lambda x: x.contact(alice)), "the object's own link is implied"),
    (lambda: Builders.Contact().addresses(lambda x: x.address(downing).since('2020')), 'not a property of the relation'),
    (lambda: Builders.Contact().addresses(lambda x: x.label('home')).create(), "the 'address' link is never set"),
]:
    try:
        attempt()
    except (AttributeError, ValueError) as error:
        print(f'{why:35} -> {type(error).__name__}: {error}')

the object's own link is implied    -> AttributeError: 'contact' is not a link this entry can set
not a property of the relation      -> AttributeError: since
the 'address' link is never set     -> ValueError: link 'address' is not set


## What you learned

- There are no lists or nested records. Collections are **relations** whose entries link objects and carry
  properties.
- Links are untyped names. Objects declare adjacencies with `.relations(lambda adj: adj.name(..).of(..).me(..))`.
  Schemas are defined in phases: objects, relations, adjacencies.
- Entries are added from an object: `.addresses(lambda x: x.address(obj).label('home'))`. A link takes an object or
  a Spec that builds one.
- An entry is one fact visible from every end. Entries form a set, and duplicates are elided.
- Data is read by visiting. `toolkit.entries` is a ten-line visitor.

**Next:** case study 3 edits relations (removing entries, moving house, cloning a contact) and looks at what
`update()` really replaces.